# Лабораторна робота 10.3
## Тема: Перенесення навчання (Transfer Learning) та Fine Tuning для набору даних Коти-Собаки

**Завдання:**
1. Підготувати набір даних: 2000 train, 500 val, 500 test для кожного класу.
2. Використати VGG16 для Transfer Learning.
3. Застосувати Fine Tuning.
4. Порівняти результати.

In [ ]:
import os
import zipfile
import random
import tensorflow as tf
from tensorflow.keras.optimizers import RMSprop
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from shutil import copyfile
from tensorflow.keras.applications.vgg16 import VGG16
from tensorflow.keras import layers
from tensorflow.keras import Model

# Якщо ви використовуєте Colab, завантажте датасет з Kaggle або використовуйте цей (filtered version)
# !wget --no-check-certificate https://storage.googleapis.com/mledu-datasets/cats_and_dogs_filtered.zip -O /tmp/cats_and_dogs_filtered.zip

# Створення директорій
base_dir = '/tmp/cats-v-dogs'
if not os.path.exists(base_dir):
    os.mkdir(base_dir)

train_dir = os.path.join(base_dir, 'training')
validation_dir = os.path.join(base_dir, 'testing')
test_dir = os.path.join(base_dir, 'final_test') # Окремий тестовий набір

train_cats_dir = os.path.join(train_dir, 'cats')
train_dogs_dir = os.path.join(train_dir, 'dogs')
validation_cats_dir = os.path.join(validation_dir, 'cats')
validation_dogs_dir = os.path.join(validation_dir, 'dogs')
test_cats_dir = os.path.join(test_dir, 'cats')
test_dogs_dir = os.path.join(test_dir, 'dogs')

for directory in [train_dir, validation_dir, test_dir, train_cats_dir, train_dogs_dir, 
                  validation_cats_dir, validation_dogs_dir, test_cats_dir, test_dogs_dir]:
    if not os.path.exists(directory):
        os.makedirs(directory)

In [ ]:
# Функція для розділення даних
def split_data(SOURCE, TRAINING, TESTING, FINAL_TEST, SPLIT_SIZE_TRAIN, SPLIT_SIZE_VAL):
    files = []
    for filename in os.listdir(SOURCE):
        file = SOURCE + filename
        if os.path.getsize(file) > 0:
            files.append(filename)
        else:
            print(filename + " is zero length, so ignoring.")

    training_length = int(len(files) * SPLIT_SIZE_TRAIN)
    validation_length = int(len(files) * SPLIT_SIZE_VAL)
    # Rest goes to final test

    shuffled_set = random.sample(files, len(files))
    training_set = shuffled_set[0:training_length]
    validation_set = shuffled_set[training_length:(training_length + validation_length)]
    final_test_set = shuffled_set[(training_length + validation_length):]

    for filename in training_set:
        this_file = SOURCE + filename
        destination = TRAINING + filename
        copyfile(this_file, destination)

    for filename in validation_set:
        this_file = SOURCE + filename
        destination = TESTING + filename
        copyfile(this_file, destination)
        
    for filename in final_test_set:
        this_file = SOURCE + filename
        destination = FINAL_TEST + filename
        copyfile(this_file, destination)

# Приклад виклику (шляхи треба замінити на реальні, де лежать всі коти і всі собаки)
# CAT_SOURCE_DIR = "/tmp/PetImages/Cat/"
# DOG_SOURCE_DIR = "/tmp/PetImages/Dog/"
# split_data(CAT_SOURCE_DIR, train_cats_dir, validation_cats_dir, test_cats_dir, 0.8, 0.1)

## Transfer Learning з VGG16

In [ ]:
# Завантажуємо VGG16
pre_trained_model = VGG16(input_shape=(150, 150, 3), include_top=False, weights='imagenet')

for layer in pre_trained_model.layers:
    layer.trainable = False

# pre_trained_model.summary()

last_layer = pre_trained_model.get_layer('block5_pool')
last_output = last_layer.output

x = layers.Flatten()(last_output)
x = layers.Dense(512, activation='relu')(x)
x = layers.Dropout(0.3)(x)
x = layers.Dense(1, activation='sigmoid')(x)

model = Model(pre_trained_model.input, x)

model.compile(optimizer = RMSprop(learning_rate=0.0001), 
              loss = 'binary_crossentropy', 
              metrics = ['accuracy'])

# Генератори
train_datagen = ImageDataGenerator(
      rescale = 1./255.,
      rotation_range=40,
      width_shift_range=0.2,
      height_shift_range=0.2,
      shear_range=0.2,
      zoom_range=0.2,
      horizontal_flip=True,
      fill_mode='nearest')

test_datagen = ImageDataGenerator(rescale = 1./255.)

train_generator = train_datagen.flow_from_directory(train_dir,
                                                    batch_size = 20,
                                                    class_mode = 'binary', 
                                                    target_size = (150, 150))

validation_generator =  test_datagen.flow_from_directory(validation_dir,
                                                          batch_size  = 20,
                                                          class_mode  = 'binary', 
                                                          target_size = (150, 150))

history = model.fit(
            train_generator,
            validation_data = validation_generator,
            steps_per_epoch = 100,
            epochs = 10,
            validation_steps = 25,
            verbose = 2)

## Fine Tuning для VGG16

In [ ]:
from tensorflow.keras.optimizers import SGD

# Розморожуємо останній блок (block5)
pre_trained_model.trainable = True
set_trainable = False
for layer in pre_trained_model.layers:
    if layer.name == 'block5_conv1':
        set_trainable = True
    if set_trainable:
        layer.trainable = True
    else:
        layer.trainable = False

model.compile(optimizer=SGD(learning_rate=0.00001, momentum=0.9),
              loss='binary_crossentropy',
              metrics=['accuracy'])

history_fine = model.fit(
      train_generator,
      steps_per_epoch = 100,
      epochs = 10,
      validation_data = validation_generator,
      validation_steps = 25,
      verbose=2)

## Порівняння та Тестування

In [ ]:
import matplotlib.pyplot as plt
acc = history.history['accuracy']
val_acc = history.history['val_accuracy']
loss = history.history['loss']
val_loss = history.history['val_loss']

epochs = range(len(acc))

plt.plot(epochs, acc, 'r', label='Training accuracy')
plt.plot(epochs, val_acc, 'b', label='Validation accuracy')
plt.title('Training and validation accuracy')
plt.legend(loc=0)
plt.figure()
plt.show()